# Project01 · Diabetes Multi Agent

Code thực nằm trực tiếp trong cell. Các module backend là export của những cell này; chỉnh thuật toán ở notebook rồi chạy `notebook_source.py export`. Notebook 01 chứa model/retrieval, 02 chứa agent/API, 03 chứa evaluator/plot. Không cần mở `.py` để xem implementation. `%%project_module` thực thi source trong cell ở namespace riêng; không đọc implementation từ file khác.

Mặc định không train, không TEST cũ, không tự chạy evaluation. Kết quả bên dưới là demo development hoặc đọc output đã lưu.

In [1]:
from pathlib import Path
import sys, os, types, __future__
from IPython.core.magic import register_cell_magic
PROJECT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'datasets/assets.json').exists() and (p/'notebooks').exists())
sys.path.insert(0,str(PROJECT/'web/backend'))
os.environ['OPENBLAS_NUM_THREADS']='1'
RUN_TRAINING=False
RUN_TEST=False
RUN_EVALUATION=False
@register_cell_magic
def project_module(line,cell):
    name=line.strip()
    parent,_,child=name.rpartition('.')
    if parent not in sys.modules:
        package=types.ModuleType(parent);package.__path__=[str(PROJECT/'web/backend/project01_core')];sys.modules[parent]=package
    module=sys.modules.get(name)
    if module is None:
        module=types.ModuleType(name);module.__package__=parent;module.__file__=str(PROJECT/'web/backend'/Path(*name.split('.')).with_suffix('.py'));sys.modules[name]=module;setattr(sys.modules[parent],child,module)
    exec(compile(cell,'<project01-notebook>/'+name,'exec',flags=__future__.annotations.compiler_flag,dont_inherit=True),module.__dict__)


## agents

Coordinator, Risk/Knowledge Agent, Verifier, Reporter; budget và process deadline.

In [2]:
%%project_module project01_core.agents
"""Five bounded roles; action traces, real tools, evidence checks and hard process deadline."""



In [3]:
%%project_module project01_core.agents
import copy



In [4]:
%%project_module project01_core.agents
import math



In [5]:
%%project_module project01_core.agents
import multiprocessing as mp



In [6]:
%%project_module project01_core.agents
import time



In [7]:
%%project_module project01_core.agents
import uuid



In [8]:
%%project_module project01_core.agents
from project01_core.foundation import LIMITATION, digest, focus_terms, plain, probability, tokenize, validate_profile



In [9]:
%%project_module project01_core.agents
VARIANTS=('P0','P1','P2')



### validate_request

In [10]:
%%project_module project01_core.agents
def validate_request(request):
    if not isinstance(request,dict) or set(request)-{'mode','question','profile','language'}:raise ValueError('Unexpected request fields')
    if request.get('mode') not in {'knowledge','numeric','combined'}:raise ValueError('Unknown mode')
    if request.get('language','en') not in {'en','vi'}:raise ValueError('Unknown language')
    question=request.get('question','')
    if not isinstance(question,str) or not 1<=len(question.strip())<=500 or any(ord(c)<32 for c in question):raise ValueError('Question must be 1–500 plain-text characters')
    profile=request.get('profile')
    if profile is not None and (not isinstance(profile,dict) or len(profile)>21):raise ValueError('Invalid profile object')
    return {**request,'language':request.get('language','en')}



### policy_reason

In [11]:
%%project_module project01_core.agents
def policy_reason(question):
    normalized=plain(question)
    refused=['ignore all','ignore your','ignore previous','guaranteed cure','cure for all','guaranteed profitable','recommend a guaranteed','insulin dose','antibiotic dose','personal insulin','issue a diagnosis','diagnose me','chan doan cho toi','ke don','lieu insulin','lieu thuoc','bo qua quy tac','tiet lo bi mat','reveal secrets']
    if any(term in normalized for term in refused):return 'request_outside_academic_scope'
    if any(term in normalized for term in ['price of gold','investment','apartment price','sale price','gia vang','gia nha','football']):return 'non_diabetes_domain'
    return None



### plan_request

In [12]:
%%project_module project01_core.agents
def plan_request(request):
    return {'need_risk':request['mode'] in {'numeric','combined'},'need_knowledge':request['mode'] in {'knowledge','combined'},'mode':request['mode'],'policy':policy_reason(request['question'])}



### verify_outputs

In [13]:
%%project_module project01_core.agents
def verify_outputs(runtime,request,risk,knowledge):
    """Separate numeric consistency, citation integrity and heuristic claim support."""
    errors=[];checks={};want=plan_request(request)
    if risk is not None:
        score=risk.get('score')
        correct=isinstance(score,(int,float)) and math.isfinite(score) and 0<=score<=1
        correct=correct and math.isclose(score,probability(risk['logit']),rel_tol=1e-6,abs_tol=1e-6)
        correct=correct and risk['profile_sha256']==digest(request['profile']) and risk['model_sha256']==runtime.model_hash
        checks['numeric_consistency']=bool(correct)
        if not correct:errors.append('numeric_mismatch')
    if knowledge is not None:
        evidence={row['chunk_id']:row for row in knowledge['evidence']}
        integrity=True
        for key,row in evidence.items():
            original=runtime.chunks.get(key)
            integrity=integrity and original is not None and original['text']==row['text'] and original['document_id']==row['document_id']
        checks['citation_integrity']=bool(integrity)
        cited=[evidence.get(key) for key in knowledge['citations']]
        support=bool(knowledge['answer'] and cited and all(row is not None for row in cited))
        focus=focus_terms(knowledge['query_normalized'])
        for row in cited:
            if row is None:continue
            support=support and knowledge['answer'] in row['text']
            support=support and (not focus or len(focus&set(tokenize(knowledge['answer'])))>=max(1,math.ceil(len(focus)*0.5)))
        checks['verbatim_focused_claim_support']=bool(support)
        if not integrity:errors.append('citation_integrity')
        if not support:errors.append('insufficient_or_unsupported_evidence')
    if want['need_risk'] and risk is None:checks['numeric_available']=False
    if want['need_knowledge'] and knowledge is None:checks['knowledge_available']=False
    return {'status':'passed' if not errors else 'rejected','checks':checks,'errors':errors,'support_method':'verbatim span + topic-token heuristic; not expert semantic entailment'}



### compose_answer

In [14]:
%%project_module project01_core.agents
def compose_answer(status,risk,knowledge,validation,errors,variant):
    sections=[]
    if risk is not None:
        sections.append({'title':'Đầu ra mô hình khảo sát','text':f"Score {risk['score']:.6f}; lớp {risk['class_id']}: {risk['class_name']}. Không phải xác suất mắc mới đã hiệu chuẩn."})
    if knowledge and knowledge['answer']:
        sections.append({'title':'Bằng chứng truy hồi · tiếng Anh','text':knowledge['answer']})
    if validation and not validation['valid']:
        sections.append({'title':'Cần bổ sung hồ sơ','text':'Thiếu: '+', '.join(validation['missing'])+'; sai: '+', '.join(validation['invalid']+validation['extra'])})
    if errors:sections.append({'title':'Giới hạn xử lý','text':'; '.join(errors)})
    return {'status':status,'sections':sections,'variant':variant,'limitation':LIMITATION,'synthesis':'verified_sections' if variant=='P2' else 'unverified_tool_outputs'}



### assist_inline

In [15]:
%%project_module project01_core.agents
def assist_inline(runtime,request,variant='P2',*,fault=None,deadline_seconds=30):
    request=validate_request(request)
    if variant not in VARIANTS:raise ValueError('Unknown variant')
    if fault not in {None,'missing_index','model_unavailable','citation_not_supporting','altered_numeric_value','deadline','empty_retrieval','transient'}:raise ValueError('Undeclared internal fault')
    start=time.perf_counter();deadline=start+deadline_seconds;trace=[];calls=0;errors=[]
    request_id='p01_'+uuid.uuid4().hex
    def event(agent,tool,started,output,outcome='ok'):
        trace.append({'node_id':f'n{len(trace)+1}','agent':agent,'tool':tool,'parent_ids':[trace[-1]['node_id']] if trace else [],'duration_ms':(time.perf_counter()-started)*1000,'outcome':outcome,'output':output})
    def tool_call(agent,name,function):
        nonlocal calls
        for attempt in range(2):
            if calls>=8 or time.perf_counter()>=deadline:raise TimeoutError('request_budget')
            tick=time.perf_counter();calls+=1
            if fault=='transient' and attempt==0:
                event(agent,name,tick,{'error':'transient','attempt':attempt},'error');continue
            value=function();event(agent,name,tick,{'result_sha256':digest(value),'attempt':attempt},'ok');return value
        raise RuntimeError('Tool retry budget exceeded')
    tick=time.perf_counter();plan=plan_request(request)
    event('Coordinator' if variant!='P0' else 'Direct','plan_request',tick,plan)
    risk=None;knowledge=None;validation=None;verification={'status':'not_run','checks':{}}
    if plan['policy']:
        errors.append(plan['policy']);status='refused'
    else:
        if fault=='deadline':time.sleep(deadline_seconds+0.3)
        if plan['need_risk']:
            tick=time.perf_counter();validation=validate_profile(request.get('profile'),runtime.schema)
            event('Risk Agent','validate_profile',tick,validation,'ok' if validation['valid'] else 'clarification')
            if validation['valid']:
                if fault=='model_unavailable':errors.append('model_unavailable')
                else:risk=tool_call('Risk Agent','predict_cdc',lambda:runtime.predict(request['profile']))
        if plan['need_knowledge']:
            if fault=='missing_index':errors.append('index_unavailable')
            else:
                knowledge=tool_call('Knowledge Agent','retrieve_medical',lambda:runtime.retrieve(request['question'],request['language']))
                if fault=='empty_retrieval':knowledge.update(answer='',citations=[],evidence=[],ranked_chunk_ids=[])
        if fault=='altered_numeric_value' and risk:risk['score']=0.99 if risk['score']<0.5 else 0.01
        if fault=='citation_not_supporting' and knowledge and knowledge['answer']:
            wrong=next(row for row in knowledge['evidence'] if knowledge['answer'] not in row['text']) if any(knowledge['answer'] not in r['text'] for r in knowledge['evidence']) else None
            if wrong:knowledge['citations']=[wrong['chunk_id']]
            else:knowledge['answer']='Unsupported universal cure claim.'
        if variant=='P2':
            tick=time.perf_counter();verification=verify_outputs(runtime,request,risk,knowledge)
            event('Verifier','check_outputs',tick,verification,'ok' if verification['status']=='passed' else 'rejected')
            if 'insufficient_or_unsupported_evidence' in verification['errors']:
                tick=time.perf_counter();event('Verifier','request_more_evidence',tick,{'remaining_loops':0,'reason':'missing_focused_support'})
                knowledge=tool_call('Knowledge Agent','retrieve_medical_expanded',lambda:runtime.retrieve(request['question'],request['language'],True))
                if fault=='empty_retrieval':knowledge.update(answer='',citations=[],evidence=[],ranked_chunk_ids=[])
                tick=time.perf_counter();verification=verify_outputs(runtime,request,risk,knowledge)
                event('Verifier','check_outputs_after_retrieval',tick,verification,'ok' if verification['status']=='passed' else 'rejected')
            if 'numeric_mismatch' in verification['errors']:risk=None;errors.append('numeric_mismatch_rejected')
            if any(e in verification['errors'] for e in ['citation_integrity','insufficient_or_unsupported_evidence']):
                errors.append('unsupported_knowledge_rejected')
                if knowledge:knowledge['answer']='';knowledge['citations']=[]
        if validation and not validation['valid']:status='clarification' if not knowledge or not knowledge['answer'] else 'partial'
        elif errors:status='unavailable' if not risk and not(knowledge and knowledge['answer']) else 'partial'
        elif plan['need_knowledge'] and not(knowledge and knowledge['answer']):status='insufficient_evidence' if risk is None else 'partial'
        else:status='answered'
    tick=time.perf_counter();answer=compose_answer(status,risk,knowledge,validation,errors,variant)
    event('Reporter' if variant=='P2' else 'Direct','compose',tick,{'status':status,'section_count':len(answer['sections'])})
    if time.perf_counter()>deadline:raise TimeoutError('request_budget')
    return {'request_id':request_id,'variant':variant,'status':status,'risk':risk,'knowledge':knowledge,'profile_validation':validation,'verification':verification,'answer':answer,'errors':errors,'trace':trace,'tool_calls':calls,'retrieval_loops':sum(node['tool']=='retrieve_medical_expanded' for node in trace),'latency_ms':(time.perf_counter()-start)*1000,'limitations':[LIMITATION],'input_sha256':digest(request)}



### _child_send

In [16]:
%%project_module project01_core.agents
def _child_send(connection,runtime,request,variant,fault,deadline_seconds):
    try:connection.send({'result':assist_inline(runtime,request,variant,fault=fault,deadline_seconds=deadline_seconds)})
    except Exception as exc:connection.send({'error':type(exc).__name__})
    finally:connection.close()



### assist

In [17]:
%%project_module project01_core.agents
def assist(runtime,request,variant='P2',*,fault=None,deadline_seconds=30):
    """Hard deadline with terminate/join; no lingering tool process after timeout."""
    validate_request(request)
    if not 0<deadline_seconds<=30:raise ValueError('Invalid deadline')
    if variant not in VARIANTS:raise ValueError('Invalid variant')
    started=time.perf_counter();context=mp.get_context('fork');parent,child=context.Pipe(duplex=False)
    process=context.Process(target=_child_send,args=(child,runtime,copy.deepcopy(request),variant,fault,deadline_seconds))
    process.start();child.close()
    response=None
    try:
        if parent.poll(max(0,deadline_seconds-(time.perf_counter()-started))):
            try:response=parent.recv()
            except EOFError:response=None
    finally:
        parent.close()
        if process.is_alive():process.terminate()
        process.join(timeout=1)
        if process.is_alive():process.kill();process.join()
    elapsed=(time.perf_counter()-started)*1000
    if response and 'result' in response:
        result=response['result'];result['latency_ms']=elapsed;result['worker_terminated']=not process.is_alive();return result
    timeout=response is None or response.get('error')=='TimeoutError'
    return {'request_id':'p01_'+uuid.uuid4().hex,'variant':variant,'status':'timeout' if timeout else 'unavailable','risk':None,'knowledge':None,'profile_validation':None,'verification':{'status':'not_run','checks':{}},'answer':{'status':'timeout' if timeout else 'unavailable','sections':[],'limitation':LIMITATION},'errors':['deadline' if timeout else 'worker_error'],'trace':[],'trace_status':'not_received','tool_calls':None,'retrieval_loops':None,'latency_ms':elapsed,'worker_terminated':not process.is_alive(),'limitations':[LIMITATION],'input_sha256':digest(request)}



## transport

API Flask nhận dữ liệu có kiểm schema; cùng pipeline notebook, không giả lập.

In [18]:
%%project_module project01_core.transport
"""Local/Render Flask transport; only allowlisted JSON, no uploads or arbitrary tools."""



In [19]:
%%project_module project01_core.transport
from collections import defaultdict, deque



In [20]:
%%project_module project01_core.transport
from pathlib import Path



In [21]:
%%project_module project01_core.transport
import os

import hashlib
import json


In [22]:
%%project_module project01_core.transport
import threading



In [23]:
%%project_module project01_core.transport
import time



In [24]:
%%project_module project01_core.transport
from flask import Flask, jsonify, request, send_from_directory



In [25]:
%%project_module project01_core.transport
from project01_core.agents import VARIANTS, assist, validate_request



In [26]:
%%project_module project01_core.transport
from project01_core.foundation import Runtime, file_sha, read_json



### create_app

In [27]:
%%project_module project01_core.transport
def create_app(project_root=None):
    root=Path(project_root or Path(__file__).resolve().parents[3])
    app=Flask(__name__,static_folder=None);app.config['MAX_CONTENT_LENGTH']=16*1024
    runtime=None;lock=threading.Lock();busy=threading.BoundedSemaphore(1);rates=defaultdict(deque)
    allowed={value.strip().rstrip('/') for value in os.environ.get('WEB_ORIGIN','http://127.0.0.1:5179,http://localhost:5179,http://127.0.0.1:8031,http://localhost:8031').split(',') if value.strip()}
    def get_runtime():
        nonlocal runtime
        with lock:
            if runtime is None:
                exports=read_json(root/'artifacts/phase2/source_exports.json')
                for row in exports['files']:
                    if file_sha(root/row['export'])!=row['sha256']:raise ValueError('Source export mismatch')
                    notebook=read_json(root/row['notebook'])
                    fingerprint=hashlib.sha256(json.dumps([(c['cell_type'],c['source']) for c in notebook['cells']],ensure_ascii=False).encode()).hexdigest()
                    if fingerprint!=row['notebook_source_sha256']:raise ValueError('Notebook source differs from export')
                runtime=Runtime(root/'artifacts/package/v1')
        return runtime
    @app.before_request
    def guard():
        origin=request.headers.get('Origin','').rstrip('/')
        if origin and origin not in allowed and origin!=request.host_url.rstrip('/'):
            return jsonify(error={'code':'origin_not_allowed','message':'Origin is not configured'}),403
        if request.method=='OPTIONS':return '',204
        if request.path=='/api/v1/assist':
            now=time.monotonic();address=request.remote_addr or 'local'
            if address not in rates and len(rates)>=1024:rates.clear()
            bucket=rates[address]
            while bucket and bucket[0]<now-60:bucket.popleft()
            if len(bucket)>=60:return jsonify(error={'code':'rate_limit','message':'Please retry shortly'}),429
            bucket.append(now)
    @app.after_request
    def headers(response):
        origin=request.headers.get('Origin','').rstrip('/')
        if origin in allowed:
            response.headers['Access-Control-Allow-Origin']=origin;response.headers['Vary']='Origin'
        response.headers['Access-Control-Allow-Methods']='GET,POST,OPTIONS'
        response.headers['Access-Control-Allow-Headers']='Content-Type'
        response.headers['X-Content-Type-Options']='nosniff'
        response.headers['Referrer-Policy']='same-origin'
        response.headers['Cache-Control']='no-store' if request.path.startswith('/api/') else 'no-cache'
        return response
    @app.errorhandler(413)
    def too_large(_):return jsonify(error={'code':'request_too_large','message':'JSON limit is 16 KiB'}),413
    @app.errorhandler(400)
    def malformed(_):return jsonify(error={'code':'invalid_json','message':'Expected a valid JSON object'}),400
    @app.get('/health')
    def health():return jsonify(status='ok',project='project01')
    @app.get('/ready')
    def ready():
        try:
            value=get_runtime();return jsonify(status='ready',chunks=len(value.chunks),graph_anchors=len(value.graph),backend='numpy_cpu')
        except (OSError,ValueError,ImportError,KeyError):return jsonify(status='unavailable',code='artifact_or_source_unavailable'),503
    @app.get('/api/v1/examples')
    def examples():
        return jsonify(profiles=read_json(root/'datasets/demo_profiles.json')['profiles'],schema=read_json(root/'datasets/profile_schema.json'),questions=[{'title':'Kiến thức · vận động','question':'Explain physical activity and diabetes.','language':'en','mode':'knowledge'},{'title':'Kết hợp model + kiến thức','question':'Explain physical activity and diabetes.','language':'en','mode':'combined'},{'title':'Tiếng Việt · phạm vi glossary','question':'Bệnh thận tiểu đường','language':'vi','mode':'knowledge'}])
    @app.get('/api/v1/results')
    def historical_results():
        path=root/'artifacts/phase2/summary.json'
        return jsonify(read_json(path) if path.exists() else {'status':'not_evaluated','rows':[]})
    @app.post('/api/v1/assist')
    def run_assist():
        if not request.is_json:return jsonify(error={'code':'content_type','message':'Use application/json'}),415
        document=request.get_json()
        if not isinstance(document,dict):return jsonify(error={'code':'invalid_request','message':'JSON object required'}),422
        document=dict(document);variant=document.pop('variant','P2')
        try:
            validate_request(document)
            if variant not in VARIANTS:raise ValueError('Unknown variant')
        except (ValueError,TypeError):return jsonify(error={'code':'invalid_request','message':'Check mode, question, language and profile fields'}),422
        if not busy.acquire(blocking=False):return jsonify(error={'code':'busy','message':'One CPU request at a time; retry shortly'}),429
        try:
            try:result=assist(get_runtime(),document,variant)
            except (OSError,ValueError,ImportError,KeyError):return jsonify(error={'code':'artifact_unavailable','message':'Verified local assets/source are unavailable'}),503
            return jsonify(result)
        finally:busy.release()
    @app.route('/',defaults={'path':''})
    @app.route('/<path:path>')
    def frontend(path):
        if path.startswith('api/'):
            return jsonify(error={'code':'not_found','message':'Unknown API route'}),404
        folder=root/'web/frontend/dist'
        candidate=(folder/path).resolve()
        if path and candidate.is_relative_to(folder.resolve()) and candidate.is_file():return send_from_directory(folder,path)
        if (folder/'index.html').exists():return send_from_directory(folder,'index.html')
        return jsonify(project='project01',frontend='Build frontend first; see README',health='/health',ready='/ready')
    return app



## Demo development: cả hai specialist + verifier + reporter

Không chạy bộ evaluation. Mẫu có nguồn TRAIN, query là development; trace chỉ ghi hành động/tool, không chain-of-thought.

In [28]:
from project01_core.foundation import Runtime,read_json
from project01_core.agents import assist
runtime=Runtime(PROJECT/'artifacts/package/v1')
profile=read_json(PROJECT/'datasets/demo_profiles.json')['profiles'][0]['profile']
request={'mode':'combined','question':'Explain physical activity and diabetes.','profile':profile,'language':'en'}
result=assist(runtime,request,'P2')
display(result['answer'])
display(result['trace'])
assert result['risk'] is not None and result['knowledge'] is not None and result['tool_calls']<=8


{'status': 'answered',
 'sections': [{'title': 'Đầu ra mô hình khảo sát',
   'text': 'Score 0.395520; lớp 0: no diabetes. Không phải xác suất mắc mới đã hiệu chuẩn.'},
  {'title': 'Bằng chứng truy hồi · tiếng Anh',
   'text': 'Always talk with a doctor before starting a new physical activity program.'}],
 'variant': 'P2',
 'limitation': 'Học thuật trên snapshot; score nhãn khảo sát, không phải chẩn đoán hoặc xác suất mắc mới. Trích đoạn gốc tiếng Anh.',
 'synthesis': 'verified_sections'}

[{'node_id': 'n1',
  'agent': 'Coordinator',
  'tool': 'plan_request',
  'parent_ids': [],
  'duration_ms': 0.20575100006681168,
  'outcome': 'ok',
  'output': {'need_risk': True,
   'need_knowledge': True,
   'mode': 'combined',
   'policy': None}},
 {'node_id': 'n2',
  'agent': 'Risk Agent',
  'tool': 'validate_profile',
  'parent_ids': ['n1'],
  'duration_ms': 0.16465399949083803,
  'outcome': 'ok',
  'output': {'valid': True, 'missing': [], 'invalid': [], 'extra': []}},
 {'node_id': 'n3',
  'agent': 'Risk Agent',
  'tool': 'predict_cdc',
  'parent_ids': ['n2'],
  'duration_ms': 1.9621500005087,
  'outcome': 'ok',
  'output': {'result_sha256': '08ae1fa0992f33de6fc8c4eb93c40c786f9f2eb2e3f569660422e5fc6ad68fa5',
   'attempt': 0}},
 {'node_id': 'n4',
  'agent': 'Knowledge Agent',
  'tool': 'retrieve_medical',
  'parent_ids': ['n3'],
  'duration_ms': 23.152945000219916,
  'outcome': 'ok',
  'output': {'result_sha256': '0034bf236f3fce66296d956cf1bf6c08e50632fb37efcf01c992622e78177764',
 